# Data preprocessing

In order to develop a proper model for the estimation of the surface water fracture, we first must adapt the data of previous satelites so that they can be used in our model.

### Library loading

Due to discrepancies between the rasterio and gdal libraries, we'll be using two different environments in this notebook.

#### General libraries

These are the necessary libraries to run most of the code.

In [1]:
import xarray as xr
import numpy as np
import os
import zipfile
import io
import re
import requests

from collections import defaultdict
from pathlib import Path
from tqdm import tqdm
from typing import List
from datetime import datetime, date, timedelta

time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

#### gdal library

The gdal library is necessary for the reprojection part of the notebook.

In [2]:
import os
import re
from osgeo import gdal, osr

ModuleNotFoundError: No module named 'osgeo'

## Data download

We'll start by manually downloading all of the [WINDSAT](https://data.remss.com/TB/intercalibration/windsat_TB_maps_daily_025deg_unfiltered/) satellite data. The [LPDR](http://files.ntsg.umt.edu/data/LPDR_v3/GeoTif/2017/) data can be downloaded directly without issues.

Note that the download speed is excrutiengly long (around 15 minutes per file). It is recomended to download it in chuncks, progressively moving the `end_date` further away, or obtain the data in any other way.

In [23]:
BASE_URL = (
    "https://data.remss.com/TB/intercalibration/"
    "windsat_TB_maps_daily_025deg_unfiltered/"
)

OUTPUT_DIR = os.path.join("data", "WINDSAT")
os.makedirs(OUTPUT_DIR, exist_ok=True)

start_date = date(2017, 1, 1)
end_date = date(2017, 12, 31)

current_date = start_date

while current_date <= end_date:
    datestr = current_date.strftime("%Y_%m_%d")
    filename = f"RSS_WINDSAT_DAILY_TBTOA_MAPS_{datestr}.nc"
    url = BASE_URL + filename
    local_path = os.path.join(OUTPUT_DIR, filename)

    if os.path.exists(local_path):
        print(f"Skipping (already exists): {filename}")
        current_date += timedelta(days=1)
        continue

    print(f"Downloading: {filename}")

    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()

        with open(local_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

    except requests.RequestException as e:
        print(f"Failed to download {filename}: {e}")

    current_date += timedelta(days=1)

print("Download process completed.")

Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_01.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_02.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_03.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_04.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_05.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_06.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_07.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_08.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_09.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_10.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_11.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_12.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_13.nc
Skipping (already exists): RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_14.nc
Skipping (already ex

## Data analysis

Now that we have all of our data, let's check it's structure and see if it's properly formated.

### LPDR

Let's check the current grid of these files

In [2]:
lpdr_ds = xr.open_dataset("data/LPDR/2017tif/AMSRU_Mland_2017001A.tif", decode_times=time_coder)

print("Dimensions:", dict(lpdr_ds.sizes))
print("Coordinates:")
for coord in ["x", "y"]:
    if coord in lpdr_ds.coords:
        print(f"  {coord}: {lpdr_ds.coords[coord].values[0]}  →  {lpdr_ds.coords[coord].values[-1]}")

Dimensions: {'band': 7, 'x': 1383, 'y': 586}
Coordinates:
  x: -17321659.775000002  →  17321659.77499638
  y: 7332251.062494965  →  -7332251.0625035055


The LPDR dataset has an EASE-v2 projection.

We can also analyze the general structure of the dataset.

In [11]:
lpdr_ds

<xarray.Dataset> Size: 23MB
Dimensions:      (band: 7, x: 1383, y: 586)
Coordinates:
  * band         (band) int64 56B 1 2 3 4 5 6 7
  * x            (x) float64 11kB -1.732e+07 -1.73e+07 ... 1.73e+07 1.732e+07
  * y            (y) float64 5kB 7.332e+06 7.307e+06 ... -7.307e+06 -7.332e+06
    spatial_ref  int64 8B ...
Data variables:
    band_data    (band, y, x) float32 23MB ...

Creating a codebook of these file will help us keep track of the different variables later on.

| Band | Band name | Parameter | Description                                                                 | Unit        | Valid range |
|------|-----------|-----------|------------------------------------------------------------------------------|-------------|-------------|
| 1    | fw        | —         | 30-day smoothed open water fraction                                          | Dimensionless | 0–1         |
| 2    | fwns      | —         | Non-smoothed open water fraction                                             | Dimensionless | 0–1         |
| 3    | Tmn / Tmx | —         | Daily surface air temperature minima or maxima, corresponding to descending or ascending pass retrievals | Kelvin      | 240–340     |
| 4    | PWV       | —         | Vertically integrated atmospheric water vapor                                | mm          | 0–80        |
| 5    | VOD       | —         | Vegetation optical depth at 10.7 GHz                                         | Neper       | 0–3         |
| 6    | vsm       | —         | Volumetric soil moisture at 10.7 GHz                                         | cm³/cm³     | 0–1         |
| 7    | VPD       | —         | Near-surface atmospheric vapor pressure deficit                              | kPa         | ≥ 0         |

### Windsat bands

Again, we'll observe the current grid.

In [31]:
windsat_ds = xr.open_dataset("data/WINDSAT/RSS_WINDSAT_DAILY_TBTOA_MAPS_2017_01_01.nc", decode_times=time_coder)
windsat_ds = windsat_ds.set_coords(["longitude", "latitude"])

print("Dimensions:", dict(windsat_ds.sizes))
print("Coordinates:")
for coord in ["longitude", "latitude"]:
    if coord in windsat_ds.coords:
        print(f"  {coord}: {windsat_ds.coords[coord].values[0]}  →  {windsat_ds.coords[coord].values[-1]}")

Dimensions: {'longitude_grid': 1440, 'latitude_grid': 720, 'swath_sector': 2, 'look_direction': 2, 'frequency_band': 5, 'polarization': 4, 'polarization_dual': 2}
Coordinates:
  longitude: 0.125  →  359.875
  latitude: -89.875  →  89.875


The WINDSAT dataset has a simple EQR projection.

This dataset is quite large, so a preliminary analisis of its variables is near mandatory.

In [32]:
windsat_ds

<xarray.Dataset> Size: 1GB
Dimensions:         (longitude_grid: 1440, latitude_grid: 720, swath_sector: 2,
                     look_direction: 2, frequency_band: 5, polarization: 4,
                     polarization_dual: 2)
Coordinates:
    longitude       (longitude_grid) float32 6kB 0.125 0.375 ... 359.6 359.9
    latitude        (latitude_grid) float32 3kB -89.88 -89.62 ... 89.62 89.88
Dimensions without coordinates: longitude_grid, latitude_grid, swath_sector,
                                look_direction, frequency_band, polarization,
                                polarization_dual
Data variables: (12/32)
    node            (swath_sector) int32 8B ...
    look            (look_direction) int32 8B ...
    frequency_vpol  (frequency_band) float32 20B ...
    frequency_hpol  (frequency_band) float32 20B ...
    eia_nominal     (frequency_band) float32 20B ...
    time            (frequency_band, latitude_grid, longitude_grid, look_direction, swath_sector) object 166MB ...
    ...              ...
    colcld_WSAT     (latitude_grid, longitude_grid, swath_sector) float32 8MB ...
    rain_IMERG      (latitude_grid, longitude_grid, swath_sector) float32 8MB ...
    rain_WSAT       (latitude_grid, longitude_grid, swath_sector) float32 8MB ...
    tran            (polarization_dual, frequency_band, latitude_grid, longitude_grid, look_direction, swath_sector) float32 166MB ...
    tbup            (polarization_dual, frequency_band, latitude_grid, longitude_grid, look_direction, swath_sector) float32 166MB ...
    tbdw            (polarization_dual, frequency_band, latitude_grid, longitude_grid, look_direction, swath_sector) float32 166MB ...
Attributes: (12/71)
    Conventions:                            CF-1.7
    title:                                  RSS WindSat TOA Brightness Temper...
    version:                                V01.0
    summary:                                The dataset contains RSS WindSat ...
    references:                              [1] T. Meissner et al., Remote S...
    acknowledgement:                        Funded under NASA Grant 80NSSC21K...
    ...                                     ...
    Source_of_ancillary_NOAA_OI_SST:        NOAA OI SST V2 High Resolution Da...
    Source_of_ancillary_IMERG_rain_rate:    Huffman, G. et al.,  2019. NASA G...
    Source_of_ancillary_CCMP_wind:          Mears, C. et al., 2023.Remote Sen...
    Source_of_ancillary_HYCOM_SSS:          Hybrid Coordinate Ocean Model, GL...
    Source_of_ancillary_ERA5:               ECMWF Reanalysis v5 (ERA5). https...
    Source_of_RSS_WindSat_AS_ECV:           https://www.remss.com/missions/wi...

The following codebook collects the information provided in the summary.

| Variable         | Category           | Description                                    | Units             | Data Type | Notes                                      |
| ---------------- | ------------------ | ---------------------------------------------- | ----------------- | --------- | ------------------------------------------ |
| `longitude`      | Geolocation        | Geodetic longitude of observation              | degrees east      | float     | Range typically [-180, 180]                |
| `latitude`       | Geolocation        | Geodetic latitude of observation               | degrees north     | float     | Range [-90, 90]                            |
| `node`           | Orbit geometry     | Orbital node of satellite swath                | categorical / int | integer   | Ascending or descending orbit              |
| `look`           | Viewing geometry   | Look direction                                 | unitless          | integer   | 0 = fore, 1 = aft                          |
| `frequency_vpol` | Instrument         | Center frequency of V-polarization channel     | GHz               | float     | Per frequency band                         |
| `frequency_hpol` | Instrument         | Center frequency of H-polarization channel     | GHz               | float     | Per frequency band                         |
| `eia_nominal`    | Viewing geometry   | Nominal Earth incidence angle                  | degrees           | float     | Per frequency band                         |
| `time`           | Temporal           | Time of observation                            | seconds           | float     | Seconds since 2000-01-01 00:00:00 UTC      |
| `eaa`            | Viewing geometry   | Boresight Earth azimuth angle                  | degrees           | float     | Range [0, 360]                             |
| `eia`            | Viewing geometry   | Boresight Earth incidence angle                | degrees           | float     | Range [0, 90]                              |
| `tbtoa`          | Radiometry         | Top-of-atmosphere brightness temperature       | Kelvin (K)        | float     | Observed microwave brightness temperature  |
| `quality_flag`   | Quality control    | Quality control bitmask                        | unitless          | uint32    | 32-bit encoded quality and screening flags |
| `sss_HYCOM`      | Ocean (HYCOM)      | Sea surface salinity from HYCOM                | PSU               | float     | Model-derived                              |
| `surtep_REY`     | Ocean (Reynolds)   | Sea surface temperature (NOAA Reynolds V2 OI)  | Kelvin (K)        | float     | Daily gridded OI SST                       |
| `fland_06`       | Surface            | Land fraction at 6 GHz                         | unitless          | float     | Fractional land contamination              |
| `fland_10`       | Surface            | Land fraction at 10 GHz                        | unitless          | float     | Fractional land contamination              |
| `surtep_WSAT`    | Windsat            | Skin temperature                               | Kelvin (K)        | float     | Windsat V8 product                         |
| `colvap_WSAT`    | Windsat            | Column-integrated water vapor                  | kg m⁻²            | float     | Windsat V8 product                         |
| `colcld_WSAT`    | Windsat            | Column-integrated cloud liquid water           | kg m⁻²            | float     | Windsat V8 product                         |
| `winspd_WSAT`    | Windsat            | Sea surface wind speed                         | m s⁻¹             | float     | Windsat V8 product                         |
| `rain_WSAT`      | Windsat            | Surface rain rate                              | mm hr⁻¹           | float     | Windsat V8 product                         |
| `winspd_CCMP`    | CCMP               | Sea surface wind speed                         | m s⁻¹             | float     | Cross-Calibrated Multi-Platform product    |
| `windir_CCMP`    | CCMP               | Sea surface wind direction                     | degrees           | float     | Meteorological convention                  |
| `surtep_ERA5`    | ERA5               | Surface (skin) temperature                     | Kelvin (K)        | float     | ERA5 reanalysis                            |
| `airtep_ERA5`    | ERA5               | 2 m air temperature                            | Kelvin (K)        | float     | ERA5 near-surface air temperature          |
| `colvap_ERA5`    | ERA5               | Column-integrated water vapor                  | kg m⁻²            | float     | ERA5 reanalysis                            |
| `colcld_ERA5`    | ERA5               | Column-integrated cloud liquid water           | kg m⁻²            | float     | ERA5 reanalysis                            |
| `winspd_ERA5`    | ERA5               | 10 m neutral-stability wind speed              | m s⁻¹             | float     | ERA5 reanalysis                            |
| `windir_ERA5`    | ERA5               | 10 m wind direction                            | degrees           | float     | ERA5 reanalysis                            |
| `surtep_CMC`     | CMC                | Sea surface temperature                        | Kelvin (K)        | float     | Canadian Meteorological Centre SST         |
| `rain_IMERG`     | IMERG              | Surface rain rate                              | mm hr⁻¹           | float     | IMERG V6 precipitation product             |
| `tran`           | Radiative transfer | Total atmospheric transmittance                | unitless          | float     | Dual-polarization dimension                |
| `tbdw`           | Radiative transfer | Atmospheric downwelling brightness temperature | Kelvin (K)        | float     | Dual-polarization dimension                |
| `tbup`           | Radiative transfer | Atmospheric upwelling brightness temperature   | Kelvin (K)        | float     | Dual-polarization dimension                |

## Data projection

Since both datasets do not share the same grid, we need to reproject them to a common grid. We'll do this by reprojecting the LPDR dataset to the same projection as the Windsat dataset.

In [11]:
def reproject_file(file_path: str, output_folder: str = None) -> bool:
    """
        Read the input geotiff in EASE v1
        Reproject + resample into ED 0.25º
        create latitude an longitude bands for convenience.
        (bands added as second to last, and last band respectively)

        param output_folder: name of the new file to save reprojected data.
            If None, data will be ovewritten in file_path.

        Returns bool: whether or not the file was succesfully reprojected
    """
    dataset = gdal.Open(file_path)

    # Define the geotransform for the output dataset
    target_geotransform = (0, 0.25, 0.0, 90.0, 0.0, -0.25)

    output_width = 1440
    output_height = 720

    # Define src and geotransform from the input:
    source_srs = osr.SpatialReference()
    source_srs.ImportFromEPSG(3410)

    # Desired projection
    target_srs = osr.SpatialReference()
    target_srs.ImportFromEPSG(4326)

    # Declare the output file and driver:
    driver = gdal.GetDriverByName("GTiff")

    if output_folder is None:
        output_file = os.path.join(os.path.dirname(file_path), "temp.tif")

    else:
        output_file = os.path.join(output_folder, os.path.basename(file_path))

    # Create the new dataset
    output_dataset = driver.Create(
        output_file, output_width, output_height,
        dataset.RasterCount,
        gdal.GDT_Float32
    )
    output_dataset.SetProjection(target_srs.ExportToWkt())
    output_dataset.SetGeoTransform(target_geotransform)

    # Set the output dataset value to -999.0, instead of 0.
    for i in range(1, output_dataset.RasterCount, 1):
        band = output_dataset.GetRasterBand(i)
        arr = band.ReadAsArray()
        arr = arr - 999.0
        band.WriteArray(arr)

    # Reproject and resample using gdal.Warp()
    gdal.Warp(
        output_dataset,
        dataset,
        dstSRS=target_srs.ExportToWkt(),
        width=output_width,
        height=output_height,
        resampleAlg="near",
        # GRA_Bilinear TODO: bilinear warp does not work with Nodata params,
        # there will be values between -999 and the valid range
        srcNodata=float(-999.0),
        dstNodata=-999.0,
    )

    # Close the files
    dataset = None
    output_dataset = None

    if output_folder is None:
        # Delete original file, rename temp file.
        os.remove(file_path)
        os.rename(output_file, file_path)

    return True

source_folder = 'data/LPDR/2017tif'
output_folder = 'data/LPDR/2017tifrep'

print("START")

# NOTE: Do not reproject Quality Flag files for now,
# those files end in '\d{3}QA.tif'
# Select only ascending and descending passes:
regex = r"\d{7}[AD].tif"

for file_name in os.listdir(source_folder):
    file_path = os.path.join(source_folder, file_name)
    print(f"Reprojecting {file_path}")

    outcome = reproject_file(file_path, output_folder=output_folder)

    if outcome:
        print("Reprojected")
        if output_folder is not None:
            print(
                f"""New file saved {
                    os.path.join(output_folder, file_name)
                }"""
            )
    else:
        print(f"File {file_name} is already reprojected")

print("DONE")

START
Reprojecting data/LPDR/2017tif\AMSRU_Mland_2017001A.tif


NameError: name 'gdal' is not defined

Let's check the new files to see if the reprojection worked accordingly.

### LPDR reprojected

In [2]:
lpdr_rep_ds = xr.open_dataset("data/LPDR/2017tifrep/AMSRU_Mland_2017001A.tif")

lpdr_rep_ds = lpdr_rep_ds.rename({
    "x": "longitude",
    "y": "latitude"
})
lpdr_rep_ds = lpdr_rep_ds.sortby("latitude")

print("Dimensions:", dict(lpdr_rep_ds.sizes))
print("Coordinates:")
for coord in ["longitude", "latitude"]:
    if coord in lpdr_rep_ds.coords:
        print(f"  {coord}: {lpdr_rep_ds.coords[coord].values[0]}  →  {lpdr_rep_ds.coords[coord].values[-1]}")

Dimensions: {'band': 7, 'longitude': 1440, 'latitude': 720}
Coordinates:
  longitude: 0.125  →  359.875
  latitude: -89.875  →  89.875


As we can see, the reprojection worked perfectly, and we can move on to the next step.

## Data mixing

Since all datasets now follow the same grid, we can go ahead and combine all three of them into a single training dataset. However, if we were to combine all files without previous pruning, we'll end up with unmanageble datasets. For this reason, we'll first select only the necessary data for our research.

This is the data we'll keep of every type.

#### LPDR

* Direction: Descending

* Variables: Everyone but fw

#### WINDSAT

* Direction: Descending (1)

* Frequencies: 18.7 (2 Ku) and 37.0 (4 Ka)

* Polarizations: V (0) and H (1)

* Look direction: Fore (0)

* Variables: tbtoa, tbup, tran, surtep_ERA5

### File pairing

Before merging the proper datasets together, we have to group each file by the day of the year. We can do without the ascending or quality flags LPDR files.

In [2]:
def get_day_of_year(file_path):
    file_path = str(file_path)
    if 'AMSRU' in file_path:
        year_doyy_str = os.path.basename(file_path).split('_')[2][:7]
        year_doyy = year_doyy_str[-3:]
    elif 'WINDSAT' in file_path:
        year, month, day = os.path.basename(file_path).split("_")[-3:]
        day = day.replace(".nc", "")
        date = datetime(int(year), int(month), int(day))
        year_doyy = f"{date.timetuple().tm_yday:03d}"
    return year_doyy

def group_files_by_day(lpdr_dir: str, windsat_dir: str) -> List[List[str]]:
    # day -> {"lpdr": [...], "windsat": [...]}
    grouped = defaultdict(lambda: {"lpdr": [], "windsat": []})

    # Collect LPDR files — ONLY those ending with 'D'
    for path in Path(lpdr_dir).iterdir():
        if path.is_file():
            # check basename without extension
            if path.stem.endswith("D"):
                doy = get_day_of_year(path)
                grouped[doy]["lpdr"].append(str(path))

    # Collect WINDSAT files
    for path in Path(windsat_dir).iterdir():
        if path.is_file():
            doy = get_day_of_year(path)
            grouped[doy]["windsat"].append(str(path))

    # Build final list, sorted by DOY
    output = []
    for doy in sorted(grouped.keys()):
        lpdr_files = sorted(grouped[doy]["lpdr"])
        windsat_files = sorted(grouped[doy]["windsat"])

        # Sanity checks
        if len(lpdr_files) > 4:
            raise ValueError(f"More than 1 LPDR-D files for DOY {doy}")
        if len(windsat_files) > 1:
            raise ValueError(f"More than 1 WINDSAT file for DOY {doy}")

        # Pair only when a WINDSAT file exists
        if windsat_files:
            slot = lpdr_files + windsat_files
            output.append(slot)

    return output

In [3]:
lpdr_dir = "data/LPDR/2017tifrep/"
windsat_dir = "data/WINDSAT/"

corresponding_files = group_files_by_day(lpdr_dir, windsat_dir)

### Landmask

Before proceding with the unification, let's prepare two datasets that identify different masses of water. Taht way, we can later on remove data related to oceans, since it will not enrich the models in training.

In [4]:
def lon_180_to_360(ds, lon_name="longitude_grid"):
    lon_new = ds[lon_name] % 360
    ds = ds.assign_coords({lon_name: lon_new})
    ds = ds.sortby(lon_name)
    return ds

In [5]:
landmask_ds = xr.open_dataset("data/LANDMASK/GLWD_main_class_025deg.tif", decode_times=time_coder).rename({
    "x": "longitude_grid",
    "y": "latitude_grid",
})
landmask_ds_band_data = {}
landmask_ds_band_data['clase'] = landmask_ds.band_data.sel(band=1).drop_vars(["band", "spatial_ref"])
landmask_ds_band_data = xr.Dataset(landmask_ds_band_data)
landmask_ds_band_data = lon_180_to_360(landmask_ds_band_data)
landmask_ds_band_data.to_netcdf("data/LANDMASK/GLWD_main_class_025deg.h5", format="NETCDF4")

landmask_ds_pct = xr.open_dataset("data/LANDMASK/GLWD_all_classes_area_pct_025deg.tif", decode_times=time_coder).rename({
    "x": "longitude_grid",
    "y": "latitude_grid",
})
landmask_ds_pct_band_data = {}
landmask_ds_pct_band_data['porcentaje clase'] = landmask_ds_pct.band_data.sel(band=1).drop_vars(["band", "spatial_ref"])
landmask_ds_pct_band_data = xr.Dataset(landmask_ds_pct_band_data)
landmask_ds_pct_band_data = lon_180_to_360(landmask_ds_pct_band_data)
landmask_ds_pct_band_data.to_netcdf("data/LANDMASK/GLWD_all_classes_area_pct_025deg.h5", format="NETCDF4")

### Dataset unification

For every day there are four files of LPDR data, one full dataset for every swath_sector (Ascending and Descending) and two quality flags for every respecting swath_sector.

We'll achieve a unified dataset by adding every variable of these four files into the WINDSAT one.

In [6]:
VARIABLES = {
    1: "fw",
    2: "fwns",
    3: "Tmn",
    4: "PWV",
    5: "VOD",
    6: "vsm",
    7: "VPD",
}

WINDSAT_VARS = [
    "tbtoa",
    "tbup",
    "tran",
    "surtep_ERA5"
]

In [7]:
def load_amsru(data_path):
    ds = xr.open_dataset(data_path)
    ds = ds.rename({"x": "longitude_grid", "y": "latitude_grid"})

    return ds

def extract_amsru_variables(ds, band_names):
    lpdr_vars = {}

    for band, name in band_names.items():
        lpdr_vars[name] = (
            ds.band_data
            .sel(band=band)
            .drop_vars(["band", "spatial_ref"])
        )

    return xr.Dataset(lpdr_vars)

def reduce_windsat_dataset(ds):
    # Subset using actual dimension names
    ds = ds.isel(
        swath_sector=1,          # Descending
        look_direction=0,        # Fore
        frequency_band=[2, 4],   # 18.7 & 37 GHz
        polarization_dual=[0, 1],     # V/H
    )
    return ds[WINDSAT_VARS]

def flatten_windsat_variables(ds):
    out = {}

    # --- tbtoa: frequency_band + polarization ---
    da = ds["tbtoa"]

    for fi in range(da.sizes["frequency_band"]):
        for pi in range(da.sizes["polarization"]):
            name = f"tbtoa_f{int(da.frequency_band[fi])}_p{int(da.polarization[pi])}"
            out[name] = (
                da.isel(frequency_band=fi, polarization=pi)
                  .squeeze(drop=True)
            )

    # --- tbup: frequency_band + polarization_dual ---
    da = ds["tbup"]

    for fi in range(da.sizes["frequency_band"]):
        for pi in range(da.sizes["polarization_dual"]):
            name = f"tbup_f{int(da.frequency_band[fi])}_p{int(da.polarization_dual[pi])}"
            out[name] = (
                da.isel(frequency_band=fi, polarization_dual=pi)
                  .squeeze(drop=True)
            )

    # --- tran: frequency_band + polarization_dual ---
    da = ds["tran"]

    for fi in range(da.sizes["frequency_band"]):
        for pi in range(da.sizes["polarization_dual"]):
            name = f"tran_f{int(da.frequency_band[fi])}_p{int(da.polarization_dual[pi])}"
            out[name] = (
                da.isel(frequency_band=fi, polarization_dual=pi)
                  .squeeze(drop=True)
            )

    # --- surtep_ERA5: already 2D ---
    out["surtep_ERA5"] = ds["surtep_ERA5"]

    return xr.Dataset(out)

In [8]:
output_dir = Path("data/MERGED")
output_dir.mkdir(parents=True, exist_ok=True)

landmask_ds_band_data = xr.open_dataset("data/LANDMASK/GLWD_main_class_025deg.h5", decode_times=time_coder)
landmask_ds_pct_band_data = xr.open_dataset("data/LANDMASK/GLWD_all_classes_area_pct_025deg.h5", decode_times=time_coder)

for files in tqdm(corresponding_files, desc="Processing days", unit="day"):
    lpdr_D_data, windsat_file = files

    doy = get_day_of_year(windsat_file)
    out_file = output_dir / f"WINDSAT_AMSRU_2017_{doy}.nc"
    
    ds_D = windsat_ds = merged_ds = None

     # --- Load AMSRU D ---
    ds_D = load_amsru(lpdr_D_data)
    amsru_D = extract_amsru_variables(ds_D, VARIABLES)

    windsat_ds = xr.open_dataset(windsat_file, decode_times=time_coder)
    windsat_ds = reduce_windsat_dataset(windsat_ds)
    windsat_ds = flatten_windsat_variables(windsat_ds)

    # --- Merge (NO QA) ---
    merged_ds = xr.merge([windsat_ds, amsru_D], join="exact")
    merged_ds = xr.merge([merged_ds, landmask_ds_band_data], join="exact")
    merged_ds = xr.merge([merged_ds, landmask_ds_pct_band_data], join="exact")

    merged_ds.to_netcdf(out_file, format="NETCDF4")

Processing days: 100%|██████████| 362/362 [09:51<00:00,  1.64s/day]


Let's finish off by checking that everything went well.

In [9]:
windsat_ds = xr.open_dataset("data/MERGED/WINDSAT_AMSRU_2017_001.nc", decode_times=time_coder)
windsat_ds

<xarray.Dataset> Size: 116MB
Dimensions:           (latitude_grid: 720, longitude_grid: 1440)
Coordinates:
  * latitude_grid     (latitude_grid) float64 6kB 89.88 89.62 ... -89.62 -89.88
  * longitude_grid    (longitude_grid) float64 12kB 0.125 0.375 ... 359.6 359.9
Data variables: (12/26)
    tbtoa_f0_p0       (latitude_grid, longitude_grid) float32 4MB ...
    tbtoa_f0_p1       (latitude_grid, longitude_grid) float32 4MB ...
    tbtoa_f0_p2       (latitude_grid, longitude_grid) float32 4MB ...
    tbtoa_f0_p3       (latitude_grid, longitude_grid) float32 4MB ...
    tbtoa_f1_p0       (latitude_grid, longitude_grid) float32 4MB ...
    tbtoa_f1_p1       (latitude_grid, longitude_grid) float32 4MB ...
    ...                ...
    PWV               (latitude_grid, longitude_grid) float32 4MB ...
    VOD               (latitude_grid, longitude_grid) float32 4MB ...
    vsm               (latitude_grid, longitude_grid) float32 4MB ...
    VPD               (latitude_grid, longitude_grid) float32 4MB ...
    clase             (latitude_grid, longitude_grid) float64 8MB ...
    porcentaje clase  (latitude_grid, longitude_grid) float64 8MB ...